In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed

### future 对象

- Future 的生命周期和状态转换
    - 生命周期：submit → running → done → result
    - `future.cancel()` 只能取消 还在排队 的任务。
    - 对于 正在运行 的任务，cancel() 返回 False 且无任何效果。

```python
from concurrent.futures import ThreadPoolExecutor
import time

executor = ThreadPoolExecutor(max_workers=1)

def slow_task():
    time.sleep(2)
    return "Done"

future = executor.submit(slow_task)

# 状态1: PENDING（未完成）
print(future.done())  # False
print(future.running())  # True (如果已开始) 或 False (还在队列)

# 尝试取消
print(future.cancel())  # False (已经开始运行，无法取消)

# 状态2: RUNNING（运行中）
# 此时 cancel() 返回 False
# 此时 result() 会阻塞

# 状态3: FINISHED（完成）
time.sleep(3)
print(future.done())  # True
print(future.result())  # "Done"

# 状态4: CANCELLED（已取消）
# 只有在任务开始前调用 cancel() 才能成功

# 状态5: EXCEPTION（异常）
def error_task():
    raise ValueError("Oops")

future2 = executor.submit(error_task)
try:
    future2.result()
except ValueError as e:
    print(f"Caught: {e}")
```

```python
from concurrent.futures import ThreadPoolExecutor
import time

class ActionGenAsyncStat:
    llm_future = None
    think_done = 0

executor = ThreadPoolExecutor(max_workers=2)

def llm_generate(instruction):
    """模拟 LLM 生成"""
    time.sleep(2)
    ActionGenAsyncStat.think_done = 1  # 思考完成
    return f"Action for: {instruction}"

# 迭代 N
instruction = "拿起苹果"
ActionGenAsyncStat.llm_future = executor.submit(llm_generate, instruction)
print("Submitted LLM task")

# 检查是否完成
if not ActionGenAsyncStat.llm_future.done():
    print("LLM still running, returning NoneAction")
    # 返回占位动作

# 迭代 N+1
time.sleep(2.5)
if ActionGenAsyncStat.llm_future.done():
    result = ActionGenAsyncStat.llm_future.result()
    print(f"Got result: {result}")
```

In [1]:
import threading
from concurrent.futures import ThreadPoolExecutor

class SimplePoolManager:
    def __init__(self, max_workers=5):
        self._executor = None
        self._lock = threading.Lock()

    def get_executor(self):
        with self._lock:
            if self._executor is None:
                self._executor = ThreadPoolExecutor(max_workers=5)
            return self._executor

    def submit(self, fn, *args, **kwargs):
        executor = self.get_executor()
        return executor.submit(fn, *args, **kwargs)

# 使用
manager = SimplePoolManager()
future = manager.submit(lambda x: x**2, 10)
print(future.result())  # 100

100


1. **单例模式**：全局只有一个线程池，避免线程泛滥
2. **懒初始化**：第一次使用时才创建 executor
3. **线程安全**：用锁保护 executor 创建

### code examples


- 并行查询多个 API

```python
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def query_api(api_name, delay):
    """模拟 API 调用"""
    time.sleep(delay)
    return f"{api_name} result"

def parallel_query():
    apis = [
        ("OpenAI", 2),
        ("Cerebras", 1),
        ("Memory", 0.5),
    ]

    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {
            executor.submit(query_api, name, delay): name
            for name, delay in apis
        }

        # 按完成顺序处理
        for future in as_completed(futures, timeout=3):
            api_name = futures[future]
            try:
                result = future.result()
                print(f"{api_name}: {result}")
            except Exception as e:
                print(f"{api_name} failed: {e}")

parallel_query()
# 输出顺序: Memory → Cerebras → OpenAI
```


```python
pbar = None
if self.show_progress:
    pbar = tqdm(
        total=len(supported_files),
        desc=f"Processing files ({self.parser_type})",
        unit="file",
    )

try:
    with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
        # Submit all tasks
        future_to_file = {
            executor.submit(
                self.process_single_file,
                file_path,
                output_dir,
                parse_method,
                **kwargs,
            ): file_path
            for file_path in supported_files
        }

        # Process completed tasks
        for future in as_completed(
            future_to_file, timeout=self.timeout_per_file
        ):
            success, file_path, error_msg = future.result()

            if success:
                successful_files.append(file_path)
            else:
                failed_files.append(file_path)
                errors[file_path] = error_msg

            if pbar:
                pbar.update(1)

except Exception as e:
    self.logger.error(f"Batch processing failed: {str(e)}")
    # Mark remaining files as failed
    for future in future_to_file:
        if not future.done():
            file_path = future_to_file[future]
            failed_files.append(file_path)
            errors[file_path] = f"Processing interrupted: {str(e)}"
            if pbar:
                pbar.update(1)
```